# WX · Experiment 0 · Sensor-space PSD / DE classical baselines (LR, SVM) (matched wMNE)

The identical experiment as the LCMV notebook (`configs/experiments/lx/lx_exp0_sensor_psd_de.yaml`; the wMNE configs inherit it unchanged), on the matched
wMNE stores. - **4 variant(s):** `logreg__psd`, `logreg__de`, `svm__psd`, `svm__de`
- **Settings:** xu_dev_cv, xu_holdout (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 1 repeats × 5-fold subject-level outer CV; inner 5-fold subject-level CV over 5 hyperparameter combination(s); refit on the full training fold.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** matched wMNE counterpart of the D3 LCMV / Xu suite (methodology log entry 55, `docs/wmne_xu.md`). All
outputs are under `<outputs>/D3_WMNE_XU/`, with its own code pin `D3_WMNE_XU/CODE_COMMIT.txt`. The LCMV suite
(`D3_LCMV_XU/`: stores, results, pins), `graphsD3_v4_*` and D1/D2 are only read, never written. Safe to run while
LCMV experiments run in another Colab session (separate folders and pin; one GPU per session).
**Validation:** the frozen LCMV / Xu protocol, verified before any unit runs: the same 65 participants and labels,
the 52 development / 13 holdout participants of the LCMV `xu_split.json`, identical five development folds; the
store must have passed validation and the match against the LCMV store. The holdout is never used for selection.

**Runs both modes:** A (`configs/experiments/wx/tenfold/wx10_exp0_sensor_psd_de.yaml`) first, then B (`configs/experiments/wx/wx_exp0_sensor_psd_de.yaml`). Resumable. Results:
`<outputs>/D3_WMNE_XU/experiments_A_xu_tenfold/` and `.../experiments_B_all_windows/`. CPU is sufficient.

In [ ]:
# ===== D3 wMNE / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_WX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The wMNE / Xu suite has its own code pin, <outputs>/D3_WMNE_XU/CODE_COMMIT.txt (all WX notebooks run
# that commit); the LCMV suite pin D3_LCMV_XU/CODE_COMMIT.txt and all other pins are only reported.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'WX']
     + (['--update-code'] if UPDATE_WX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
WX = OUT + '/D3_WMNE_XU'
print('suite outputs:', WX, '| LCMV reference (read only):', LX)

## 1. Status (from the checkpoints on Drive)

In [ ]:
CONFIGS = {'A': 'configs/experiments/wx/tenfold/wx10_exp0_sensor_psd_de.yaml', 'B': 'configs/experiments/wx/wx_exp0_sensor_psd_de.yaml'}
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')
for mode, cfg in CONFIGS.items():
    print('==== mode', mode)
    sh(f"python scripts/experiment.py status --config {cfg}")

## 2. Run mode A, then mode B (resumable)

In [ ]:
ERRORS = {}
for mode, cfg in CONFIGS.items():
    try:
        sh(f"python scripts/experiment.py run --config {cfg}")
    except RuntimeError as exc:
        ERRORS[mode] = exc
        print(f'mode {mode}: some units failed; details after the results')

## 3. Results (saved outputs only) and the paired wMNE vs LCMV table

In [ ]:
import glob
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 300, 'display.width', 250)
for mode, cfg in CONFIGS.items():
    sh(f"python scripts/experiment.py report --config {cfg}")
    root = 'experiments_A_xu_tenfold' if mode == 'A' else 'experiments_B_all_windows'
    reports = sorted(glob.glob(f'{WX}/{root}/{Path(cfg).stem}/_reports/report_*'))
    if reports:
        print('==== mode', mode, reports[-1])
        display(pd.read_csv(reports[-1] + '/results_summary.csv'))
sh("python scripts/wx_compare.py")
if ERRORS:
    raise RuntimeError(f'units did not complete in mode(s) {sorted(ERRORS)}; completed units are saved. Run all again.')